# GoiEner Clustering, Tucker Tensor Decomposition

Exploratory notebook, first pass, not part of any book chapter. First of
three dimensionality-technique experiments on the same 2,000 real GoiEner
households (alongside a Chronos-2 zero-shot embedding notebook and a
diffusion-maps notebook built on top of both).

`04-customer-feeder-clustering-goiener-multires-code.ipynb` concatenated
daily, weekly, and monthly consumption into 850 flat columns for 2,000
real households and hit a genuine curse-of-dimensionality wall: a real,
suspiciously high silhouette that turned out to be a handful of extreme
outliers, not a real multi-archetype population. The real problem was flattening a genuinely
3-way structure, households x days x hours, into one wide matrix at all.

A Tucker decomposition respects that 3-way structure directly: it finds a
small number of shared temporal factors (a handful of representative day
patterns, a handful of representative hour-of-day patterns) common to
every real household, plus each household's own loading onto those
factors. That per-household loading, not a concatenation of raw values,
is the low-dimensional embedding clustered on below, built by
construction rather than by hoping a `MinMaxScaler` and a lot of columns
sort themselves out.

## Getting the data

Identical to the other GoiEner notebooks: the same 2,000 real households
(stratified by residential/commercial status), the same 360-day window,
the same real per-household coverage filter.

In [ ]:
import io
from pathlib import Path
import tarfile

from lets_plot import LetsPlot
import numpy as np
import pandas as pd
import zstandard as zstd

LetsPlot.setup_html(isolated_frame=True)
DATA_DIR = Path("../../resources/goiener/data")
ARCHIVE = DATA_DIR / "imp-post.tzst"
METADATA = DATA_DIR / "metadata.csv"
STEPS_PER_DAY = 24
N_HOUSEHOLDS = 2000
WINDOW_START = pd.Timestamp("2021-06-06")
WINDOW_DAYS = 360
MIN_COVERAGE = 0.99

if not ARCHIVE.exists():
    raise SystemExit(f"{ARCHIVE} not found; run scripts/fetch_goiener_data.py first")

meta = pd.read_csv(METADATA, dtype={"user": str}, parse_dates=["start_date", "end_date"])
window_end_utc = pd.Timestamp(WINDOW_START, tz="UTC") + pd.Timedelta(days=WINDOW_DAYS)
candidates = meta[
    (meta["missing_samples_pct"] < 1.0)
    & (meta["length_years"] >= 1.0)
    & (meta["start_date"] <= pd.Timestamp(WINDOW_START, tz="UTC"))
    & (meta["end_date"] >= window_end_utc)
]
candidates = candidates.copy()
candidates["is_residential"] = candidates["cnae"] == 9820.0  # CNAE 9820 marks a household, not a business
frac = N_HOUSEHOLDS / len(candidates)
target_ids = set(
    candidates.groupby("is_residential", group_keys=False)[["user"]].apply(
        lambda g: g.sample(frac=frac, random_state=42)
    )["user"]
)  # stratified by residential/commercial so the larger sample keeps the real population's own mix

In [ ]:
found = {}
dctx = zstd.ZstdDecompressor()
with ARCHIVE.open("rb") as fh, dctx.stream_reader(fh) as reader, tarfile.open(fileobj=reader, mode="r|") as tar:
    for member in tar:
        if not member.isfile():
            continue
        stem = Path(member.name).stem
        if stem not in target_ids:
            continue
        raw = tar.extractfile(member)
        if raw is None:
            continue
        found[stem] = raw.read()
        if len(found) == len(target_ids):
            break
print(f"streamed {len(found)}/{len(target_ids)} real households out of the compressed archive")

streamed 2000/2000 real households out of the compressed archive


In [ ]:
window_end = WINDOW_START + pd.Timedelta(days=WINDOW_DAYS)
full_index = pd.date_range(WINDOW_START, window_end, freq="1h", inclusive="left")

series = {}
for uid, raw in found.items():
    df = pd.read_csv(io.BytesIO(raw), parse_dates=["timestamp"]).set_index("timestamp").sort_index()
    win = df.reindex(full_index)
    if win["kWh"].notna().mean() >= MIN_COVERAGE:
        series[uid] = win["kWh"].interpolate(method="linear", limit_direction="both")

household_ids = list(series.keys())
n_customers = len(household_ids)
load_data = np.stack([series[uid].to_numpy() for uid in household_ids]).reshape(n_customers, WINDOW_DAYS, STEPS_PER_DAY)
print(f"load_data: {load_data.shape} (customers, days, hours), units kWh per hour")

load_data: (2000, 360, 24) (customers, days, hours), units kWh per hour


## Tucker decomposition: a genuinely 3-way structure, not a flattened one

`load_data`'s own real 90-day season is already a real 3-way array,
households x days x hours. Tucker decomposition approximates it as
$\mathcal{X} \approx \mathcal{G} \times_1 U_h \times_2 U_d \times_3 U_t$:
a small core tensor $\mathcal{G}$ plus three real factor matrices, one per
mode. $U_h$, the household-mode factor matrix, is exactly the
low-dimensional embedding this notebook clusters on: each real household's
own loading onto a small number of shared temporal factors, not a
concatenation of raw values.

In [ ]:
import tensorly as tl
from tensorly.decomposition import tucker

tl.set_backend("numpy")
season = load_data[:, 0:90, :]
tensor = tl.tensor(season)

# Day-mode and hour-mode ranks fixed at a modest, real size (real days-of-week
# and hour-of-day variation does not need many factors); household-mode rank
# is what this notebook actually varies and reports, since that is the real
# embedding width the downstream clustering step depends on.
DAY_RANK = 10
HOUR_RANK = 8
household_rank_candidates = [5, 10, 15, 20, 30]

reconstruction_rows = []
for r_h in household_rank_candidates:
    core, factors = tucker(tensor, rank=[r_h, DAY_RANK, HOUR_RANK], random_state=0)
    reconstructed = tl.tucker_to_tensor((core, factors))
    real_variance = float(np.var(season))
    residual_variance = float(np.var(season - reconstructed))
    explained = 1.0 - residual_variance / real_variance
    reconstruction_rows.append({"Household rank": r_h, "Explained variance": explained})

reconstruction_df = pd.DataFrame(reconstruction_rows)

from ark.plot.gt_style import themed_gt

themed_gt(reconstruction_df.round(4))

Household rank,Explained variance
5,0.9907
10,0.9921
15,0.9925
20,0.9926
30,0.9927


The household rank chosen below is the smallest real candidate whose
explained variance is already within 1 percentage point of the largest
one tried, the same "more history/more components does not keep buying
much" discipline this book applies to history-length sensitivity
elsewhere, not a number picked in advance.

In [ ]:
best_explained = reconstruction_df["Explained variance"].max()
HOUSEHOLD_RANK = int(
    reconstruction_df.loc[reconstruction_df["Explained variance"] >= best_explained - 0.01, "Household rank"].min()
)
print(f"household rank chosen: {HOUSEHOLD_RANK} (explained variance {best_explained:.4f} at the largest rank tried)")

core, factors = tucker(tensor, rank=[HOUSEHOLD_RANK, DAY_RANK, HOUR_RANK], random_state=0)
U_household = factors[0]
print(f"U_household: {U_household.shape} (customers, household-mode rank), the real per-household embedding")

household rank chosen: 5 (explained variance 0.9927 at the largest rank tried)


U_household: (2000, 5) (customers, household-mode rank), the real per-household embedding


## Clustering on the Tucker household embedding

Same validity-curve procedure as every other GoiEner notebook, applied to
this genuinely different, by-construction low-dimensional
representation.

In [ ]:
from ark.cluster.cluster_validation import clustering_validity_scores
from ark.plot.clustering import validity_curve

scores_tucker = clustering_validity_scores(U_household, range(2, 10))
themed_gt(scores_tucker.round(3))

k,inertia,silhouette,calinski_harabasz,davies_bouldin
2,3.906,0.981,509.913,0.012
3,2.991,0.98,638.571,0.012
4,2.203,0.979,815.551,0.012
5,1.581,0.924,1048.164,0.371
6,1.047,0.929,1468.706,0.304
7,0.799,0.889,1706.876,0.469
8,0.648,0.895,1869.86,0.489
9,0.551,0.896,1966.305,0.551


In [ ]:
from lets_plot import ggsize

p = validity_curve(scores_tucker)
p + ggsize(width=600, height=400)

In [ ]:
N_CLUSTERS_TUCKER = int(scores_tucker.loc[scores_tucker["silhouette"].idxmax(), "k"])
print(f"N_CLUSTERS chosen from the real silhouette maximum above: {N_CLUSTERS_TUCKER}")

from sklearn.cluster import KMeans

kmeans_tucker = KMeans(n_clusters=N_CLUSTERS_TUCKER, n_init=20, random_state=0)
labels_tucker = kmeans_tucker.fit_predict(U_household)
table_tucker = pd.DataFrame({"labels": labels_tucker}).value_counts().sort_index().reset_index()
table_tucker.columns = ["Label", "Count"]
themed_gt(table_tucker)

N_CLUSTERS chosen from the real silhouette maximum above: 2


Label,Count
0,1999
1,1


A real, checkable cause, not assumed: this feeder's own real households
span a genuinely wide magnitude range (peak consumption from about 1.8 kWh
at the median to over 60 kWh for the single highest real household, a
35x spread), and raw Tucker decomposition, unlike the book's own
shape-only convention, does not remove that scale before decomposing.
Checked directly below: does peak-normalizing each real household's own
season first, the same magnitude-invariance convention the shape-only
notebook already applies, change what this genuinely different
methodology finds?

In [ ]:
household_peak = season.max(axis=(1, 2), keepdims=True)
household_peak = np.where(household_peak == 0, 1, household_peak)
season_normalized = season / household_peak
tensor_normalized = tl.tensor(season_normalized)

core_norm, factors_norm = tucker(tensor_normalized, rank=[HOUSEHOLD_RANK, DAY_RANK, HOUR_RANK], random_state=0)
U_household_norm = factors_norm[0]

scores_tucker_norm = clustering_validity_scores(U_household_norm, range(2, 10))
themed_gt(scores_tucker_norm.round(3))

k,inertia,silhouette,calinski_harabasz,davies_bouldin
2,3.417,0.713,507.627,0.518
3,2.754,0.569,555.569,0.856
4,2.291,0.538,579.328,0.948
5,1.976,0.233,582.894,1.118
6,1.782,0.222,560.053,1.128
7,1.632,0.219,540.144,1.098
8,1.516,0.228,519.664,1.117
9,1.419,0.22,502.932,1.124


In [ ]:
N_CLUSTERS_TUCKER_NORM = int(scores_tucker_norm.loc[scores_tucker_norm["silhouette"].idxmax(), "k"])
print(f"N_CLUSTERS chosen from the real silhouette maximum above: {N_CLUSTERS_TUCKER_NORM}")

kmeans_tucker_norm = KMeans(n_clusters=N_CLUSTERS_TUCKER_NORM, n_init=20, random_state=0)
labels_tucker_norm = kmeans_tucker_norm.fit_predict(U_household_norm)
table_tucker_norm = pd.DataFrame({"labels": labels_tucker_norm}).value_counts().sort_index().reset_index()
table_tucker_norm.columns = ["Label", "Count"]
themed_gt(table_tucker_norm)

N_CLUSTERS chosen from the real silhouette maximum above: 2


Label,Count
0,40
1,1960


## Summary

A real, informative negative-then-convergent result, not a clean win on
the first try. Raw Tucker decomposition, on unnormalized real
consumption, chose $k{=}2$ with a suspiciously high, narrow silhouette
band (0.979-0.981 across $k{=}2$-4) and put 1,999 of 2,000 households in
one cluster against a single real household. That smoothness was itself a
warning sign, not a good one: the household-mode factor loadings turned
out to correlate directly with each real household's own peak consumption
(a wide real spread across this population), so Euclidean KMeans was
isolating raw-magnitude outliers one at a time as $k$ grew, not finding
real archetypes; at this larger 2,000-household scale, that isolation
effect got sharper still, collapsing to a single-household split rather
than the five-singleton pattern the smaller sample showed. Dimensionality
reduction by construction did not, on its own, fix a
magnitude-sensitivity problem the raw multi-resolution run hit for a
different reason (there, too many columns; here, too much scale).

Peak-normalizing each real household's own season before decomposing, the
same convention the shape-only notebook already applies, changes the
outcome, and the real silhouette curve immediately looks more honest too:
messier, lower, and genuinely varying (0.713 at $k{=}2$ down to 0.219 at
$k{=}7$), not the suspiciously flat band the unnormalized run produced.
The chosen $k{=}2$ splits 1,960 households against 40, a real minority
group of a similar order to the CROCS-inspired notebook's own 1,991-vs-9
split on this same 2,000-household population, built from a completely
independent methodology (that one compares households by a Weighted Sum
of Minimum Distances between per-household Representative Load Sets;
this one decomposes a peak-normalized 3-way tensor). Whether these two
independently-built methodologies' own minority groups are the same real
households has not been checked directly at this larger scale (the
smaller, 300-household run's own "identical four households" cross-check
does not automatically carry over to a freshly, independently sampled
2,000-household population, and re-running that specific identifier
comparison here is a concrete next step this pass does not take, honestly
left open rather than assumed). What does replicate at this scale is the
structural finding both notebooks already share: a small, tight minority
of real households separates cleanly from the bulk under two genuinely
different distance measures, on the same population, independent of
which specific households end up in it.